In [1]:
##### Converts raw raster on irrigation land into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # % of cropland equiped for irrigation 

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from rasterio.features import geometry_mask

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# irrigation raster
irrigation = f"{cd}/Data/Raw/Predictors/Irrigated_area_Mehta/G_AEI_2015.ASC"

# cropland raster
cropland = f"{cd}/Data/Raw/Predictors/Ag_land_ME/crop_land_ha_2020.tif"

# save paths
pixels_pct_irr = f"{cd}/Data/Clean/Predictors/Rasters/pct_cropland_irrigated.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/percent_cropland_irrigated.csv"

In [3]:
#### Run resampling for pixel scale 

### PREP 
# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()
    ref_nodata    = ref.nodata
    ref_data      = ref.read(1)

# build reference valid mask
if ref_nodata is not None:
    ref_valid = ~np.isnan(ref_data) if np.isnan(ref_nodata) else (ref_data != ref_nodata)
else:
    ref_valid = ~np.isnan(ref_data)

# reproject countries once
if countries.crs != dst_crs:
    countries = countries.to_crs(dst_crs)
countries = countries.reset_index(drop=True)


### RESAMPLE 
def resample_to_ref(src_path):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            src_nodata=src.nodata,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

irr  = resample_to_ref(irrigation)
crop = resample_to_ref(cropland)


### CALCULATE 
# compute % irrigated
# - where crop and irr are both valid: irr/crop if crop > 0, else 0 (true zero, not missing)
# - where crop or irr is missing: nan (to be gap-filled)
valid_data = ~np.isnan(crop) & ~np.isnan(irr)

pct_irrigated = np.full(dst_shape, np.nan, dtype=np.float32)
with np.errstate(invalid="ignore", divide="ignore"):
    pct_irrigated[valid_data] = np.where(
        crop[valid_data] > 0,
        irr[valid_data] / crop[valid_data],
        0.0
    )

pct_irrigated = np.clip(pct_irrigated, 0, 1).astype(np.float32)

### ALIGN AND GAP-FILL
# align to reference mask
pct_irrigated[~ref_valid] = np.nan
needs_fill = ref_valid & np.isnan(pct_irrigated)
print(f"[pct_irrigated] Pixels needing fill: {needs_fill.sum()}")

if needs_fill.any():
    country_stats = rasterstats.zonal_stats(
        countries,
        pct_irrigated,
        affine=dst_transform,
        stats=["mean"],
        nodata=np.nan,
    )
    country_means = {
        i: s["mean"] for i, s in enumerate(country_stats)
        if s["mean"] is not None
    }
    fill_array = np.full(dst_shape, np.nan, dtype=np.float32)
    for idx, row in countries.iterrows():
        mean_val = country_means.get(idx)
        if mean_val is None:
            continue
        country_mask = geometry_mask(
            [row.geometry],
            transform=dst_transform,
            invert=True,
            out_shape=dst_shape,
        )
        fill_array[needs_fill & country_mask] = mean_val
    pct_irrigated = np.where(needs_fill, fill_array, pct_irrigated).astype(np.float32)

    still_missing = ref_valid & np.isnan(pct_irrigated)
    if still_missing.any():
        print(f"  [pct_irrigated] Warning: {still_missing.sum()} pixels still unfilled. "
              f"These will be saved as NoData.")

# save 
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = pct_irrigated.copy()
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_pct_irr, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

print(f"  [pct_irrigated] Saved → {pixels_pct_irr}")

[pct_irrigated] Pixels needing fill: 1033949
  [pct_irrigated] Warning: 87 pixels still unfilled. These will be saved as NoData.
  [pct_irrigated] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/pct_cropland_irrigated.tif


In [4]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE

# sum irrigation and cropland in each vector
zonal_irr  = rasterstats.zonal_stats(gdf_proj, irrigation, stats="sum", nodata=-9999)
zonal_crop = rasterstats.zonal_stats(gdf_proj, cropland,   stats="sum", nodata=-9999)

### compute share at vector level
irr_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_irr])
crop_sums = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_crop])

with np.errstate(invalid="ignore", divide="ignore"):
    pct = np.where(
        (crop_sums > 0) & ~np.isnan(crop_sums) & ~np.isnan(irr_sums),
        irr_sums / crop_sums,
        np.nan
    )

result["pct_cropland_irrigated"] = np.clip(pct, 0, 1)

### save
result.to_csv(vectors, index=False)